In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType, TimestampType

In [ ]:
users_schema = StructType([
    StructField("user_id", LongType(), nullable=False),
    StructField("email", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("age", IntegerType(), nullable=True),
    StructField("gender", StringType(), nullable=True),
    StructField("job", StringType(), nullable=True),
    StructField("address", StringType(), nullable=True),
    StructField("signup", DateType(), nullable=True),
    StructField("created_at", TimestampType(), nullable=False),
    StructField("updated_at", TimestampType(), nullable=False),
])

In [ ]:
spark = SparkSession.builder.appName("Example Iceberg") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", "mmix") \
    .config("spark.hadoop.fs.s3a.secret.key", "mmixmmix") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop.type", "hadoop") \
    .config("spark.sql.catalog.hadoop.warehouse", "s3a://mmix-prod-dataengineer-datalakehouse/reducing/warehouse/") \
    .getOrCreate()

In [ ]:
url = f"jdbc:mysql://mysql-primary:3306/mmix?useUnicode=true&characterEncoding=utf8&useSSL=false&serverTimezone=UTC"
properties = {"user": "mmix", "password": "mmix", "driver": "com.mysql.cj.jdbc.Driver", "fetchsize": "10000"}
logical_datetime = "2026-01-18 00:00:00"
bound = spark.read.jdbc(url=url, table=f"(SELECT MIN(user_id) AS lowerBound, MAX(user_id) AS upperBound FROM users WHERE created_at >= {logical_datetime}) AS T", properties=properties).first()
lower = bound["lowerBound"]
upper = bound["upperBound"]

In [ ]:
users_raw = spark.read.jdbc(url=url, table="users", column="user_id", lowerBound=lower, upperBound=upper, numPartitions=4, properties=properties)
users = users_raw.select(
    col("user_id").cast("bigint").alias("user_id"),
    col("email").cast("string").alias("email"),
    col("name").cast("string").alias("name"),
    col("age").cast("int").alias("age"),
    col("gender").cast("string").alias("gender"),
    col("job").cast("string").alias("job"),
    col("address").cast("string").alias("address"),
    col("signup").cast("date").alias("signup"),
    col("created_at").cast("timestamp").alias("created_at"),
    col("updated_at").cast("timestamp").alias("updated_at"))

users.createOrReplaceTempView("stage_users")

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS glue.mmix.users_iceberg (
      user_id       BIGINT      NOT NULL,
      email         STRING      NOT NULL,
      name          STRING      NOT NULL,
      age           INT,
      gender        STRING,
      job           STRING,
      address       STRING,
      signup        DATE,
      created_at    TIMESTAMP   NOT NULL,
      updated_at    TIMESTAMP   NOT NULL
  ) USING iceberg
    PARTITIONED BY (days(created_at))
    TBLPROPERTIES ('format-version' = '2');
""")

In [ ]:
spark.sql("""
    MERGE INTO hadoop.mmix.users t USING stage_users s ON t.user_id = s.user_id
    WHEN MATCHED AND s.updated_at >= t.updated_at
    THEN
        UPDATE SET
            user_id    = s.user_id,
            email      = s.email,
            name       = s.name,
            age        = s.age,
            gender     = s.gender,
            job        = s.job,
            address    = s.address,
            signup     = s.signup,
            created_at = s.created_at,
            updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT *;
""")